# Paquete de cierre mensual 
## SCAFFOLD (sábado)

Hoy ensamblamos el trabajo de la semana en **un solo archivo** llenando el master template
`plantilla_paquete.xlsx` con **openpyxl**. El cálculo (GLM y chain-ladder) ya viene resuelto:
tu trabajo es el **export** (las celdas marcadas con `# TODO`).

Necesitas en la carpeta: `plantilla_paquete.xlsx` y `datos/datos.parquet`.


In [3]:
import numpy as np, pandas as pd
import statsmodels.api as sm, statsmodels.formula.api as smf
import chainladder as cl
import openpyxl
from openpyxl.utils import get_column_letter as L
import warnings; warnings.filterwarnings("ignore")

MONEY, FAC = "#,##0", "0.000"
VARS = [("cobertura",3),("sexo",2),("combustible",3),
        ("edad_conductor_cat",7),("potencia_cat",5),("nivel_bonus_cat",6)]
def tarifa_pos(start=4):
    pos, r = {}, start
    for name,cnt in VARS:
        pos[name] = (r, r+1, r+cnt); r += cnt+2
    return pos
pos = tarifa_pos()

## Parte 1 · Pricing (dado): base y factores del GLM

In [6]:
BINS = {"edad_conductor":[17,30,35,45,50,55,60,95],"potencia":[9,40,50,60,70,250],
        "nivel_bonus":[-1,0,1,2,5,10,25]}
d = pd.read_parquet("../m5t2_sesion1/datos/datos.parquet")
for col,ct in BINS.items(): d[col+"_cat"]=pd.cut(d[col],bins=ct).astype(str)
d["pure_premium"]=d["monto_total"]/d["exposicion"]
m = smf.glm("pure_premium ~ C(cobertura)+C(sexo)+C(combustible)+C(edad_conductor_cat)+"
            "C(potencia_cat)+C(nivel_bonus_cat)", d,
            family=sm.families.Tweedie(var_power=1.3, link=sm.families.links.Log()),
            var_weights=d["exposicion"]).fit()
PRIMA_BASE = float(np.exp(m.params["Intercept"]))
FACTORES = {n:[(str(x),float(np.exp(m.params.get(f"C({n})[T.{x}]",0.0))))
               for x in sorted(d[n].dropna().unique())] for n,_ in VARS}
PERFIL_REF = {n: FACTORES[n][-1][0] for n,_ in VARS}
def prima_python(p):
    f=PRIMA_BASE
    for n,_ in VARS: f*=dict(FACTORES[n])[p[n]]
    return f
print("Prima base:", round(PRIMA_BASE,2))

FileNotFoundError: [Errno 2] No such file or directory: '../m5t2_sesion1/datos/datos.parquet'

## Parte 2 · Reservas (dado): triángulo, factores, IBNR (CL) y BF

In [37]:
tri = cl.load_sample("raa"); frame = tri.to_frame(origin_as_datetime=False)
anios = [int(getattr(a,"year",a)) for a in frame.index]; AN = len(anios)
ldf = [float(x) for x in cl.Development().fit(tri).ldf_.to_frame(origin_as_datetime=False).values.flatten()]
TAIL = 1.0; M = frame.values
last, cdf, ibnr_cl = [], [], []
for j in range(AN):
    lc=M[j,AN-1-j]; c=float(np.prod(ldf[AN-1-j:]))*TAIL
    last.append(lc); cdf.append(c); ibnr_cl.append(lc*c-lc)
IBNR_TOTAL=float(np.nansum(ibnr_cl))
ibnr_bf=[np.linspace(18000,27000,AN)[j]*0.75*(1-1/cdf[j]) for j in range(AN)]
print("IBNR total CL:", round(IBNR_TOTAL))

IBNR total CL: 52135


## Parte 3 · Llenar el master template (TU TRABAJO)

Abre la plantilla y completa los `# TODO`. Pista: `ws.cell(fila, col, valor)` y
`ws.cell(...).number_format = MONEY`. Las fórmulas van como texto: `ws.cell(r,c,"=A1*B1")`.

In [ ]:
wb = openpyxl.load_workbook("plantilla_paquete.xlsx")

# --- Portada (dado) ---
po = wb["Portada"]; po["B3"]="=TODAY()"; po["B3"].number_format="dd/mm/yyyy"; po["B4"]="Alondra Abril Alarcon Alvarez"

# --- Tarifa ---
ta = wb["Tarifa"]
# TODO 1: escribe la prima base en B2 (formato MONEY)
ta["B2"] = round(PRIMA_BASE, 2)
ta["B2"].number_format = MONEY


# TODO 2: llena las tablas de factores. Para cada (name,_) in VARS:
#   tr,di,df = pos[name]; recorre FACTORES[name] y escribe nivel en col 1, factor en col 2 (FAC)
# for name,_ in VARS:

for name, _ in VARS:
    tr, di, df = pos[name]
    for fila, (nivel, factor) in enumerate(FACTORES[name]):
        ta.cell(di + fila,1, nivel)
        ta.cell(di + fila,2, float(factor)).number_format = FAC


# TODO 3: cotizador. Para i,(name,_) in enumerate(VARS): r=5+i
#   escribe el nivel inicial en (r,5), agrega el dropdown (data_validation list con A di:df),
#   y en (r,6) la fórmula =VLOOKUP(E{r}, Tarifa!$A$di:$B$df, 2, FALSE)
#   finalmente en (12,5): =B2*PRODUCT(F5:F10)

for i,(name,_) in enumerate(VARS): 
    r = 5 + i #renglon 5 mas lo que se vaya iterando
    tr, di, df = pos[name]
    rng = f"$A${di}:$B${df}"
    ta.cell(r, 5, FACTORES[name][0][0])
    dv = openpyxl.worksheet.datavalidation.DataValidation(type="list", formula1=f"=Tarifa!$A${di}:$A${df}")
    ta.add_data_validation(dv)
    dv.add(ta.cell(r,5))
    ta.cell(r,6,f"=VLOOKUP(E{r},{rng},2, FALSE)").number_format = FAC

ta.cell(12, 5, "=B2*PRODUCT(F5:F10)").number_format = MONEY 

In [39]:
# --- Reservas ---
re = wb["Reservas"]
# TODO 4: triángulo. for i in range(AN): escribe anios[i] en (6+i,1);
#   for dd in range(AN): si M[i,dd] no es NaN, escríbelo en (6+i,2+dd) con MONEY
for i in range(AN):
    re.cell(6 + i, 1, anios[i])
    for dd in range(AN):
        v = M[i,dd]
        if not np.isnan(v):
            re.cell(6+i, 2+dd, float(v)).number_format = MONEY

# TODO 5: factores. for dd in range(AN-1): escribe ldf[dd] (PRECISIÓN COMPLETA) en (19,2+dd) FAC
#   y el tail en (19,1+AN)
for dd in range(AN-1):
    re.cell(19, 2+dd, float(ldf[dd])).number_format = FAC

re.cell(19, 1 + AN, TAIL).number_format = FAC


# TODO 6: tabla IBNR como FÓRMULAS vivas. for j in range(AN): r=23+j
#   (r,1)=anios[j]; (r,2)=f"={L(11-j)}{6+j}"  (diagonal)
#   (r,3)= "=K19" si j==0 else f"=PRODUCT({L(11-j)}19:J19)*K19"
#   (r,4)=f"=B{r}*C{r}"; (r,5)=f"=D{r}-B{r}"; y (33,5)="=SUM(E23:E32)"
for j in range(AN):
    r = 23 + j; re.cell(r, 1, anios[j]); diag = f"{L(11 - j)}{6 + j}"
    re.cell(r, 2, f"={diag}").number_format = MONEY
    re.cell(r, 3, "=K19" if j == 0 else f"=PRODUCT({L(11 - j)}19:J19)*K19").number_format = FAC
    re.cell(r, 4, f"=B{r}*C{r}").number_format = MONEY
    re.cell(r, 5, f"=D{r}-B{r}").number_format = MONEY
    re.cell(33, 5, "=SUM(E23:E32)").number_format = MONEY

# TODO 7: comparativo CL vs BF. for j in range(AN): r=37+j
#   (r,1)=anios[j]; (r,2)=f"=E{23+j}"; (r,3)=ibnr_bf[j]; (r,4)=f"=C{r}-B{r}"
#   totales en fila 47
for j in range (AN):
    r = 37 + j; re.cell(r, 1, anios[j])
    re.cell(r, 2, f"=E{23 + j}").number_format = MONEY
    re.cell(r, 3, float(ibnr_bf[j])).number_format = MONEY
    re.cell(r, 4, f"=C{r}-B{r}").number_format = MONEY
re.cell(47, 2, "=SUM(B37:B46)"); re.cell(47, 3, "=SUM(C37:C46)"); re.cell(47, 4, "=C47-B47")  

<Cell 'Reservas'.D47>

In [40]:
# --- Resumen y Control ---
rs = wb["Resumen"]
# TODO 8: B3="='Tarifa'!B2"; B4="='Reservas'!E33"; B5="='Reservas'!C47"  (todas MONEY)
rs["B3"] = "='Tarifa'!B2"
rs["B4"] = "='Reservas'!E33"
rs["B5"] = "='Reservas'!C47"

for celda in ["B3", "B4", "B5"]:
    rs[celda].number_format = MONEY

co = wb["Control"]
# TODO 9 (pricing): B4 = prima_python(PERFIL_REF) (valor);
#   B5 = fórmula = PRIMA_BASE * producto de VLOOKUP("nivel", Tarifa!A_di:B_df, 2, FALSE) por var;
#   B6 = "=B5-B4"
co["B4"] = prima_python(PERFIL_REF)

formula = f"={PRIMA_BASE}"

for name, _ in VARS:

    tr, di, df = pos[name]

    nivel = PERFIL_REF[name]

    formula += (
        f'*VLOOKUP("{nivel}",'
        f'Tarifa!$A${di}:$B${df},2,FALSE)'
    )

co["B5"] = formula

co["B6"] =  "=B5-B4"

# TODO 10 (reservas): B10 = IBNR_TOTAL; B11="='Reservas'!E33"; B12="=B11-B10"

co["B10"] = IBNR_TOTAL
co["B11"] = "='Reservas'!E33"
co["B12"] = "=B11-B10"


for celda in ["B4", "B5", "B6", "B10","B11", "B12"]:
    co[celda].number_format = MONEY


## Plus (si terminas)
- Protege la hoja Tarifa dejando editables solo las celdas de selección del cotizador.
- Agrega una gráfica (factores de una variable, o el desarrollo del triángulo).
- Reproducibilidad: confirma que volver a correr todo regenera el paquete idéntico.


In [ ]:
# --- PLUS 1: proteger Tarifa y permitir editar solo las selecciones ---
from openpyxl.styles import Protection
from openpyxl.chart import BarChart, Reference

for fila in ta:
    for celda in fila:
        celda.protection = Protection(locked=True)

for i, (name, _) in enumerate(VARS):
    ta.cell(row=5+i, column=5).protection = Protection(locked=False)

ta.protection.sheet = True
# Permite seleccionar celdas desbloqueadas en la hoja protegida
ta.protection.selectUnlockedCells = False


# --- PLUS 2: gráfica de factores de cobertura ---
from openpyxl.chart import BarChart, Reference
from openpyxl.chart.label import DataLabelList

# Posiciones de la tabla de cobertura
_, di, df = pos["combustible"]

# Crear gráfica
grafica = BarChart()

# Diseño
grafica.type = "bar"  # Barras horizontales
grafica.grouping = "clustered"
grafica.style = 10

# Títulos
grafica.title = "Factores por combustible"
grafica.x_axis.title = "Factor"
grafica.y_axis.title = "Categoria de combustible"

# Seleccionar valores de Excel
datos = Reference(
    ta,
    min_col=2,
    min_row=di,
    max_row=df - 1
)

categorias = Reference(
    ta,
    min_col=1,
    min_row=di,
    max_row=df - 1
)

grafica.add_data(datos, titles_from_data=False)
grafica.set_categories(categorias)

# Quitar leyenda
grafica.legend = None

# Mostrar los valores en las barras
grafica.dLbls = DataLabelList()
grafica.dLbls.showVal = True
grafica.dLbls.numFmt = "0.000"

# Color de las barras
grafica.series[0].graphicalProperties.solidFill = "FF0000"

# Ajustar tamaño
grafica.width = 25
grafica.height = 10

# Insertar en Excel
ta.add_chart(grafica, "H5")


In [42]:
wb.save("paquete_cierre.xlsx")
print("Guardado -> paquete_cierre.xlsx. Ábrelo y revisa: cotizador calcula, IBNR sale, Control ~0., Grafica tarifa")

Guardado -> paquete_cierre.xlsx. Ábrelo y revisa: cotizador calcula, IBNR sale, Control ~0., Grafica tarifa


In [44]:
#--- PLUS 3: REPRODUCIBILIDAD DEL PAQUETE ---

import openpyxl
from pathlib import Path

# 1. Definir carpeta de destino
carpeta = Path(
    r"."
)

# Crear carpeta si no existe
carpeta.mkdir(parents=True, exist_ok=True)

# 2. Definir nombres de los archivos
original = carpeta / "paquete_cierre.xlsx"
repetido = carpeta / "paquete_cierre_repetido.xlsx"

# 3. Guardar el archivo
if not original.exists():
    wb.save(original)
    print("Primera ejecución guardada correctamente.")
else:
    wb.save(repetido)
    print("Segunda ejecución guardada correctamente.")

# 4. Función para comparar los archivos
def firma_contenido_excel(ruta):

    archivo = openpyxl.load_workbook(
        ruta,
        data_only=False
    )

    firma = {}

    for hoja in archivo.worksheets:

        firma[hoja.title] = {

            "celdas": {
                (
                    c.coordinate,
                    c.value,
                    c.number_format,
                    c.protection.locked
                )
                for fila in hoja
                for c in fila
                if c.value is not None
            },

            "graficas": len(hoja._charts),

            "validaciones": (
                len(hoja.data_validations.dataValidation)
                if hoja.data_validations else 0
            ),

            "protegida": hoja.protection.sheet
        }

    archivo.close()

    return firma


# 5. Comparar resultados
if original.exists() and repetido.exists():

    iguales = (
        firma_contenido_excel(original)
        == firma_contenido_excel(repetido)
    )

    print("\nRESULTADO DE REPRODUCIBILIDAD")
    print("¿Mismo contenido y estructura básica?:", iguales)

    if iguales:
        print("Los archivos coinciden.")
    else:
        print("Presentan diferencias.")
else:
    print("\nFalta una segunda ejecución para comparar.")

Segunda ejecución guardada correctamente.

RESULTADO DE REPRODUCIBILIDAD
¿Mismo contenido y estructura básica?: True
Los archivos coinciden.
